In [66]:
import pandas as pd
import numpy as np

In [67]:
df_nong_san = pd.read_csv("../data/raw/nong_san_raw.csv")
df_weather = pd.read_csv("../data/raw/weather_history_new.csv")
df_exchange = pd.read_csv("../data/raw/exchange_usd_history.csv")
df_fuel = pd.read_csv("../data/raw/fuel_history_new.csv")
df_commodities_api = pd.read_csv("../data/raw/commodities_api_raw_v2.csv")
df_commodities_history = pd.read_csv("../data/raw/commodities_history_v2.csv")

In [68]:
print("Nông sản:", df_nong_san.shape)
print("Thời tiết:", df_weather.shape)
print("Tỷ giá:", df_exchange.shape)
print("Nhiên liệu:", df_fuel.shape)
print("Hàng hóa API:", df_commodities_api.shape)
print("Hàng hóa History:", df_commodities_history.shape)

Nông sản: (3106, 11)
Thời tiết: (854, 14)
Tỷ giá: (359, 10)
Nhiên liệu: (2228, 8)
Hàng hóa API: (10440, 12)
Hàng hóa History: (1946, 12)


In [69]:
df_nong_san = df_nong_san.replace(["?", ".", "-", "N/A"], np.nan)

df_weather = df_weather.replace(["?", ".", "-", "N/A"], np.nan)

df_exchange = df_exchange.replace(["?", ".", "-", "N/A"], np.nan)

df_fuel = df_fuel.replace(["?", ".", "-", "N/A"], np.nan)

df_commodities_api = df_commodities_api.replace(["?", ".", "-", "N/A"], np.nan)

df_commodities_history = df_commodities_history.replace(["?", ".", "-", "N/A"], np.nan)

In [70]:
print("Nông sản:")
print(df_nong_san.isna().sum())

print("\nThời tiết:")
print(df_weather.isna().sum())

print("\nTỷ giá:")
print(df_exchange.isna().sum())

print("\nNhiên liệu:")
print(df_fuel.isna().sum())

print("\nHàng hóa API:")
print(df_commodities_api.isna().sum())

print("\nHàng hóa History:")
print(df_commodities_history.isna().sum())

Nông sản:
id               0
date             0
fetched_at       0
type             0
region           0
price            0
unit             0
source           0
source_id        0
commodity        0
change        1770
dtype: int64

Thời tiết:
id                   0
date                 0
fetched_at           0
city                 0
temperature          0
humidity             0
conditions           0
forecast           854
source               0
geo_location_id     14
latitude            14
longitude           14
target_region        0
requested_city       0
dtype: int64

Tỷ giá:
id            0
date          0
fetched_at    0
currency      0
buy           0
sell          0
transfer      0
source        0
bank          0
source_id     0
dtype: int64

Nhiên liệu:
id              0
date            0
fetched_at      0
fuel_type       0
zone          668
price           0
source          0
source_id       0
dtype: int64

Hàng hóa API:
id                0
date              0
fetched_at    

### Tạo cờ báo thiếu cho `change` — Missingness Indicator (Bước 1)

In [71]:
# Nông sản: đánh dấu dòng có giá trị change bị khuyết
# 1 = thiếu, 0 = có dữ liệu
# Chỉ tạo cờ, không thay giá trị NaN của cột change

df_nong_san["change_missing"] = df_nong_san["change"].isna().astype(int)

# Kiểm tra
print(df_nong_san["change_missing"].value_counts())
print(df_nong_san[["change", "change_missing"]].head())

change_missing
1    1770
0    1336
Name: count, dtype: int64
   change  change_missing
0     NaN               1
1     NaN               1
2     NaN               1
3     NaN               1
4     NaN               1


Kiểm tra dữ liệu phân loại

In [72]:
# Nông sản
print(df_nong_san["region"].value_counts())

# Thời tiết
print(df_weather["target_region"].value_counts())

# Tỷ giá
print(df_exchange["bank"].value_counts())

# Nhiên liệu
print(df_fuel["fuel_type"].value_counts())

# Hàng hóa API
print(df_commodities_api["unit"].value_counts())

# Hàng hóa History
print(df_commodities_history["unit"].value_counts())

region
Gia Lai              483
Đắk Lắk              483
Đắk Nông             483
GARBLED              334
Gram/lít             334
UNGARBLED            334
Lâm Đồng             316
Bà Rịa - Vũng Tàu    167
Bình Phước           167
$4,017.52              1
$4,108.45              1
$4,175.07              1
$4,455.43              1
$4,376.62              1
Name: count, dtype: int64
target_region
Lâm Đồng      182
Gia Lai       168
Đắk Lắk       168
Đắk Nông      168
Bình Phước    168
Name: count, dtype: int64
bank
VCB            180
Vietcombank    179
Name: count, dtype: int64
fuel_type
Dầu hỏa 2-K            328
DO 0,001S-V            280
DO 0.05S               280
E5 RON 92-II           280
Xăng E10 RON 95-III    208
Dầu Diesel             167
Dầu Mazut              167
Xăng E5 RON 92         167
RON 95-III              76
RON 95-V                76
Dầu DO 0,001S-V         48
Dầu DO 0,05S-II         48
Xăng E10 RON 95-V       48
Xăng E5 RON 92-II       48
Xăng RON 95-III          7
Nam

Loại bỏ khoảng trắng thừa

In [73]:
# Nông sản
df_nong_san["region"] = df_nong_san["region"].astype("string").str.strip()
df_nong_san["unit"] = df_nong_san["unit"].astype("string").str.strip()

# Thời tiết
df_weather["city"] = df_weather["city"].astype("string").str.strip()
df_weather["target_region"] = df_weather["target_region"].astype("string").str.strip()

# Tỷ giá
df_exchange["bank"] = df_exchange["bank"].astype("string").str.strip()

# Nhiên liệu
df_fuel["fuel_type"] = df_fuel["fuel_type"].astype("string").str.strip()

# Hàng hóa API
df_commodities_api["unit"] = df_commodities_api["unit"].astype("string").str.strip()

# Hàng hóa History
df_commodities_history["unit"] = df_commodities_history["unit"].astype("string").str.strip()
# Xóa cột forecast vì thiếu 100% dữ liệu
df_weather.drop(columns=["forecast"], inplace=True, errors="ignore")



C:\Users\lehuu\AppData\Local\Temp\ipykernel_9476\74245500.py:21: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_weather.drop(columns=["forecast"], inplace=True, errors="ignore")


Chuẩn hóa tên ngân hàng

In [74]:
df_exchange["bank"] = df_exchange["bank"].replace({
    "VCB": "Vietcombank"
})

print(df_exchange["bank"].value_counts())

bank
Vietcombank    359
Name: count, dtype: int64[pyarrow]


Chuẩn hóa tên nhiên liệu

In [75]:
df_fuel["fuel_type"] = df_fuel["fuel_type"].replace({
    "Dầu DO 0,001S-V": "DO 0,001S-V",
    "Xăng E5 RON 92-II": "E5 RON 92-II",
    "Xăng RON 95-III": "RON 95-III"
})

print(df_fuel["fuel_type"].value_counts())

fuel_type
DO 0,001S-V            328
Dầu hỏa 2-K            328
E5 RON 92-II           328
DO 0.05S               280
Xăng E10 RON 95-III    208
Dầu Diesel             167
Dầu Mazut              167
Xăng E5 RON 92         167
RON 95-III              83
RON 95-V                76
Dầu DO 0,05S-II         48
Xăng E10 RON 95-V       48
Name: count, dtype: int64[pyarrow]


Định dạng cột zone

In [76]:
# Tạo cờ đánh dấu dữ liệu khuyết (chỉ tạo lần đầu)
# Giữ nguyên cờ nếu chạy lại cell sau khi zone đã được điền khuyết
if "zone_missing" not in df_fuel.columns:
    df_fuel["zone_missing"] = df_fuel["zone"].isna().astype(int)

# Chuẩn hóa vùng nhiên liệu
df_fuel["zone"] = df_fuel["zone"].replace({
    1.0: "1",
    2.0: "2"
})

# Chuyển về kiểu chuỗi
df_fuel["zone"] = df_fuel["zone"].astype("string")

# Thay thế giá trị khuyết
df_fuel["zone"] = df_fuel["zone"].fillna("Không rõ")

# Kiểm tra
print(df_fuel["zone"].value_counts(dropna=False))

zone
1           780
2           780
Không rõ    668
Name: count, dtype: int64[pyarrow]


Chuẩn hóa đơn vị hàng hóa

In [77]:
df_commodities_api["unit"] = df_commodities_api["unit"].replace({
    "Đồng/kg": "VND/kg",
    "USD/tấn": "USD/tonne"
})

print(df_commodities_api["unit"].value_counts())

unit
USD/tonne     2283
CNY/tấn       1728
USc/lb        1515
VND/kg        1001
USD/60kg       980
Đồng/m3        455
Nghìn/lít      455
USD/barrel     299
Đồng/m         273
Đồng/lít       271
Đồng/m2        182
USD/ounce      182
Đồng/tấn        91
Yên/tấn         91
Đồng/viên       91
USD/Mmbtu       91
Đồng/lượng      91
USD/pound       91
Đồng/cọc        90
USD/thùng       90
MYR/tấn         90
Name: count, dtype: int64[pyarrow]


Định dạng date

In [78]:
df_nong_san["date"] = pd.to_datetime(df_nong_san["date"], format="%Y-%m-%d")

df_weather["date"] = pd.to_datetime(df_weather["date"], format="%Y-%m-%d")

df_exchange["date"] = pd.to_datetime(df_exchange["date"], format="%Y-%m-%d")

df_fuel["date"] = pd.to_datetime(df_fuel["date"], format="%Y-%m-%d")

df_commodities_api["date"] = pd.to_datetime(df_commodities_api["date"], format="%Y-%m-%d")

df_commodities_history["date"] = pd.to_datetime(df_commodities_history["date"], format="%Y-%m-%d")

Định dạng fetched_at

In [79]:
df_nong_san["fetched_at"] = pd.to_datetime(df_nong_san["fetched_at"], utc=True)

df_weather["fetched_at"] = pd.to_datetime(df_weather["fetched_at"], utc=True)

df_exchange["fetched_at"] = pd.to_datetime(df_exchange["fetched_at"], utc=True)

df_fuel["fetched_at"] = pd.to_datetime(df_fuel["fetched_at"], utc=True)

df_commodities_api["fetched_at"] = pd.to_datetime(df_commodities_api["fetched_at"], utc=True)

df_commodities_history["fetched_at"] = pd.to_datetime(df_commodities_history["fetched_at"], utc=True)

Định dạng các cột dữ liệu số

In [80]:
# Nông sản
df_nong_san[["price", "change"]] = df_nong_san[["price", "change"]].apply(
    pd.to_numeric, errors="raise"
)

# Thời tiết
df_weather[["temperature", "humidity"]] = df_weather[["temperature", "humidity"]].apply(
    pd.to_numeric, errors="raise"
)

# Tỷ giá
df_exchange[["buy", "sell", "transfer"]] = df_exchange[["buy", "sell", "transfer"]].apply(
    pd.to_numeric, errors="raise"
)

# Nhiên liệu
df_fuel["price"] = pd.to_numeric(df_fuel["price"], errors="raise")

# Hàng hóa API
df_commodities_api[["price", "open_price", "high_price", "low_price"]] = df_commodities_api[
    ["price", "open_price", "high_price", "low_price"]
].apply(pd.to_numeric, errors="raise")

# Hàng hóa History
df_commodities_history[["price", "open_price", "high_price", "low_price"]] = df_commodities_history[
    ["price", "open_price", "high_price", "low_price"]
].apply(pd.to_numeric, errors="raise")

Kiểm tra sau Data Formatting

In [81]:
print("NÔNG SẢN")
print(df_nong_san.dtypes)

print("\nTHỜI TIẾT")
print(df_weather.dtypes)

print("\nTỶ GIÁ")
print(df_exchange.dtypes)

print("\nNHIÊN LIỆU")
print(df_fuel.dtypes)

print("\nHÀNG HÓA API")
print(df_commodities_api.dtypes)

print("\nHÀNG HÓA HISTORY")
print(df_commodities_history.dtypes)

NÔNG SẢN
id                              int64
date                   datetime64[us]
fetched_at        datetime64[us, UTC]
type                              str
region                         string
price                           int64
unit                           string
source                            str
source_id                         str
commodity                         str
change                        float64
change_missing                  int64
dtype: object

THỜI TIẾT
id                               int64
date                    datetime64[us]
fetched_at         datetime64[us, UTC]
city                            string
temperature                    float64
humidity                         int64
conditions                         str
source                             str
geo_location_id                float64
latitude                       float64
longitude                      float64
target_region                   string
requested_city                     str
dty

Kiểm tra các giá trị đã chuẩn hóa

In [82]:
print("Tên ngân hàng:")
print(df_exchange["bank"].unique())

print("\nSố loại nhiên liệu:")
print(df_fuel["fuel_type"].nunique())

print("\nCác vùng nhiên liệu:")
print(df_fuel["zone"].unique())

print("\nĐơn vị hàng hóa API:")
print(df_commodities_api["unit"].unique())

print("\nKiểu dữ liệu ngày:")
print(df_nong_san["date"].dtype)

print("\nKiểu dữ liệu thời điểm thu thập:")
print(df_nong_san["fetched_at"].dtype)

Tên ngân hàng:
<ArrowStringArray>
['Vietcombank']
Length: 1, dtype: string

Số loại nhiên liệu:
12

Các vùng nhiên liệu:
<ArrowStringArray>
['1', '2', 'Không rõ']
Length: 3, dtype: string

Đơn vị hàng hóa API:
<ArrowStringArray>
[    'USc/lb',   'USD/60kg', 'USD/barrel',  'USD/tonne',   'Đồng/tấn',
    'Đồng/m3',    'Yên/tấn',    'CNY/tấn',   'Đồng/cọc',  'USD/thùng',
    'MYR/tấn',  'Nghìn/lít',  'Đồng/viên',     'VND/kg',  'USD/Mmbtu',
   'Đồng/lít',    'Đồng/m2',  'USD/ounce', 'Đồng/lượng',  'USD/pound',
     'Đồng/m']
Length: 21, dtype: string

Kiểu dữ liệu ngày:
datetime64[us]

Kiểu dữ liệu thời điểm thu thập:
datetime64[us, UTC]


# Bước 3 – Feature Engineering & Feature Transformation (Chương 2 – DS111)

**Vị trí:** phần này được thêm **ngay sau** cell cuối của Bước 2: “Kiểm tra các giá trị đã chuẩn hóa”.

- **3.1. Feature Engineering:** tạo thêm cột từ ngày tháng, tính chênh lệch từ cột có sẵn và trung bình trượt 7 quan sát.
- **3.2. Feature Transformation:** áp dụng MinMaxScaler, MaxAbsScaler, StandardScaler, RobustScaler, Log và Sqrt.

Giữ nguyên 6 DataFrame gốc (`df_nong_san`, `df_weather`, `df_exchange`, `df_fuel`, `df_commodities_api`, `df_commodities_history`). Các tập con `df_coffee_gialai` và `df_coffee_scale` chỉ dùng **để thực hành**, chưa gộp CSV, chưa mã hóa phân loại hay lựa chọn đặc trưng.


### Bước 3 – Cell 1: Tách tháng và thứ từ `date` (Feature Engineering)

`dt.month`: tháng 1–12; `dt.dayofweek`: thứ trong tuần, từ **0 = thứ Hai** đến **6 = Chủ nhật**. Đặt sau phần kiểm tra cuối của Bước 2.

In [83]:
# Nông sản
df_nong_san["month"] = df_nong_san["date"].dt.month
df_nong_san["day_of_week"] = df_nong_san["date"].dt.dayofweek

# Thời tiết
df_weather["month"] = df_weather["date"].dt.month
df_weather["day_of_week"] = df_weather["date"].dt.dayofweek

# Tỷ giá
df_exchange["month"] = df_exchange["date"].dt.month
df_exchange["day_of_week"] = df_exchange["date"].dt.dayofweek

# Nhiên liệu
df_fuel["month"] = df_fuel["date"].dt.month
df_fuel["day_of_week"] = df_fuel["date"].dt.dayofweek

# Hàng hóa API
df_commodities_api["month"] = df_commodities_api["date"].dt.month
df_commodities_api["day_of_week"] = df_commodities_api["date"].dt.dayofweek

# Hàng hóa History
df_commodities_history["month"] = df_commodities_history["date"].dt.month
df_commodities_history["day_of_week"] = df_commodities_history["date"].dt.dayofweek

print(df_nong_san[["date", "month", "day_of_week"]].head())

        date  month  day_of_week
0 2026-04-24      4            4
1 2026-04-24      4            4
2 2026-04-24      4            4
3 2026-04-24      4            4
4 2026-04-23      4            3


C:\Users\lehuu\AppData\Local\Temp\ipykernel_9476\1195777136.py:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_nong_san["day_of_week"] = df_nong_san["date"].dt.dayofweek
C:\Users\lehuu\AppData\Local\Temp\ipykernel_9476\1195777136.py:7: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_weather["day_of_week"] = df_weather["date"].dt.dayofweek
C:\Users\lehuu\AppData\Local\Temp\ipykernel_9476\1195777136.py:11: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_exchange["day_of_week"] = df_exchange["date"].dt.dayofweek
C:\Users\lehuu\AppData\Local\Temp\ipykernel_9476\1195777136.py:15: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_fuel["day_of_week"] = df_fuel["date

### Bước 3 – Cell 2: Tạo đặc trưng từ các cột số có sẵn

Tỷ giá: chênh lệch giá bán–mua. Hàng hóa: khoảng giá cao–thấp. Giá trị thiếu ở `high_price` hoặc `low_price` vẫn để là `NaN`, **không tự điền 0**.

In [84]:
# Tỷ giá: chênh lệch giá bán và giá mua
df_exchange["spread"] = df_exchange["sell"] - df_exchange["buy"]

# Hàng hóa API: biên độ giá cao nhất - thấp nhất
df_commodities_api["price_range"] = (
    df_commodities_api["high_price"] - df_commodities_api["low_price"]
)

# Hàng hóa History: biên độ giá cao nhất - thấp nhất
df_commodities_history["price_range"] = (
    df_commodities_history["high_price"] - df_commodities_history["low_price"]
)

print(df_exchange[["buy", "sell", "spread"]].head())
print(df_commodities_api[["high_price", "low_price", "price_range"]].head())

     buy   sell  spread
0  26107  26387     280
1  26107  26387     280
2  26107  26387     280
3  26107  26387     280
4  26102  26382     280
   high_price  low_price  price_range
0      278.15     276.05         2.10
1      275.20     273.30         1.90
2      290.75     288.20         2.55
3      265.95     259.85         6.10
4      268.60     261.25         7.35


### Bước 3 – Cell 3: Ví dụ trung bình giá 7 quan sát gần nhất

Chỉ chọn **cà phê nội địa tại Gia Lai**, nguồn `giacaphe.com`, cùng đơn vị `VND/kg`, để không trộn sản phẩm, vùng hay nguồn khác nhau. `rolling(7)` tính trung bình **7 bản ghi liên tiếp**, không tự động đồng nghĩa 7 ngày lịch; 6 dòng đầu khuyết là bình thường.

In [85]:
df_coffee_gialai = df_nong_san.loc[
    (df_nong_san["commodity"] == "coffee")
    & (df_nong_san["type"] == "domestic")
    & (df_nong_san["region"] == "Gia Lai")
    & (df_nong_san["source"] == "giacaphe.com")
    & (df_nong_san["unit"] == "VND/kg")
    & (df_nong_san["price"] > 0),
    ["date", "region", "source", "price"]
].copy()

df_coffee_gialai = df_coffee_gialai.sort_values("date")

df_coffee_gialai["price_ma7"] = (
    df_coffee_gialai["price"].rolling(window=7, min_periods=7).mean()
)

print("Số bản ghi:", len(df_coffee_gialai))
print("Số ngày khác nhau:", df_coffee_gialai["date"].nunique())
print(df_coffee_gialai[["date", "price", "price_ma7"]].head(9))

Số bản ghi: 161
Số ngày khác nhau: 161
         date  price     price_ma7
80 2026-05-20  86100           NaN
76 2026-05-22  86700           NaN
72 2026-05-23  87900           NaN
64 2026-05-24  87900           NaN
56 2026-05-25  87600           NaN
48 2026-05-26  87800           NaN
41 2026-05-27  88500  87500.000000
32 2026-05-28  87700  87728.571429
24 2026-05-29  89100  88071.428571


### Bước 3 – Cell 4: Chuẩn bị dữ liệu minh họa cho Scaling

`price` trong `df_nong_san` có thể thuộc nhiều sản phẩm và đơn vị, một vài dòng có giá âm. Vì vậy ta dùng **tập con cùng sản phẩm, cùng đơn vị và giá dương** đã lọc ở Cell 3; tuyệt đối không áp Log/Sqrt cho toàn bộ cột giá nông sản một cách máy móc.

In [86]:
from sklearn.preprocessing import (
    MinMaxScaler,
    MaxAbsScaler,
    StandardScaler,
    RobustScaler,
    FunctionTransformer
)

df_coffee_scale = df_coffee_gialai[["date", "price"]].copy()

print("Giá nhỏ nhất:", df_coffee_scale["price"].min())
print("Giá lớn nhất:", df_coffee_scale["price"].max())
print("Có giá <= 0:", (df_coffee_scale["price"] <= 0).any())

Giá nhỏ nhất: 84000
Giá lớn nhất: 98800
Có giá <= 0: False


### Bước 3 – Cell 5: MinMaxScaler

Công thức: $v=(x-\min)/(\max-\min)$. Với dữ liệu này, đưa giá trị về đoạn **[0, 1]**.

In [87]:
mms = MinMaxScaler()
df_coffee_scale["price_mms"] = mms.fit_transform(
    df_coffee_scale[["price"]]
).ravel()

print(df_coffee_scale[["price", "price_mms"]].head())

    price  price_mms
80  86100   0.141892
76  86700   0.182432
72  87900   0.263514
64  87900   0.263514
56  87600   0.243243


### Bước 3 – Cell 6: MaxAbsScaler

Công thức theo giá trị tuyệt đối lớn nhất: $v=x/\max(|x|)$. Với dữ liệu toàn số dương, giá trị nằm trong **[0, 1]**.

In [88]:
mas = MaxAbsScaler()
df_coffee_scale["price_mas"] = mas.fit_transform(
    df_coffee_scale[["price"]]
).ravel()

print(df_coffee_scale[["price", "price_mas"]].head())

    price  price_mas
80  86100   0.871457
76  86700   0.877530
72  87900   0.889676
64  87900   0.889676
56  87600   0.886640


### Bước 3 – Cell 7: StandardScaler

Công thức: $z=(x-\mu)/\sigma$. Kết quả có trung bình gần 0 và độ lệch chuẩn gần 1; **không đảm bảo dữ liệu có phân phối chuẩn**.

In [89]:
ss = StandardScaler()
df_coffee_scale["price_ss"] = ss.fit_transform(
    df_coffee_scale[["price"]]
).ravel()

print(df_coffee_scale[["price", "price_ss"]].head())

    price  price_ss
80  86100 -1.652178
76  86700 -1.499010
72  87900 -1.192676
64  87900 -1.192676
56  87600 -1.269259


### Bước 3 – Cell 8: RobustScaler

Công thức: $v=(x-Q_2)/(Q_3-Q_1)$; dùng median và IQR nên ít nhạy với ngoại lệ hơn thang đo dựa vào min/max.

In [90]:
rs = RobustScaler()
df_coffee_scale["price_rs"] = rs.fit_transform(
    df_coffee_scale[["price"]]
).ravel()

print(df_coffee_scale[["price", "price_rs"]].head())

    price  price_rs
80  86100 -1.153846
76  86700 -1.061538
72  87900 -0.876923
64  87900 -0.876923
56  87600 -0.923077


### Bước 3 – Cell 9: Log Transform và Square Root Transform

Đúng ví dụ từ slide: `FunctionTransformer(np.log1p)` tính $\log(1+x)$ và `FunctionTransformer(np.sqrt)` tính $\sqrt{x}$. Giá của tập con đều **dương**, nên dùng được cả hai phép biến đổi.

In [91]:
log_ft = FunctionTransformer(np.log1p, validate=True)
df_coffee_scale["price_log"] = log_ft.fit_transform(
    df_coffee_scale[["price"]]
).ravel()

sqrt_ft = FunctionTransformer(np.sqrt, validate=True)
df_coffee_scale["price_sqrt"] = sqrt_ft.fit_transform(
    df_coffee_scale[["price"]]
).ravel()

print(df_coffee_scale[["price", "price_log", "price_sqrt"]].head())

    price  price_log  price_sqrt
80  86100  11.363276  293.428015
76  86700  11.370221  294.448637
72  87900  11.383966  296.479342
64  87900  11.383966  296.479342
56  87600  11.380548  295.972972


### Bước 3 – Cell 10: Kiểm tra kết quả của 6 phép chuyển đổi

Các cột `_mms`, `_mas`, `_ss`, `_rs`, `_log`, `_sqrt` được tạo để **so sánh cách biến đổi**, không thay thế cột `price` gốc. Lúc này chưa học mô hình hoặc Feature Selection.

In [92]:
columns_to_check = [
    "price", "price_mms", "price_mas", "price_ss",
    "price_rs", "price_log", "price_sqrt"
]

print(df_coffee_scale[columns_to_check].head())
print("\nThống kê sau chuyển đổi:")
print(df_coffee_scale[columns_to_check].describe().round(3))

print("\nMinMax khoảng [0, 1]:",
      df_coffee_scale["price_mms"].min() >= -1e-10
      and df_coffee_scale["price_mms"].max() <= 1 + 1e-10)
print("Các cột mới có giá trị NaN:")
print(df_coffee_scale[columns_to_check].isna().sum())

    price  price_mms  price_mas  price_ss  price_rs  price_log  price_sqrt
80  86100   0.141892   0.871457 -1.652178 -1.153846  11.363276  293.428015
76  86700   0.182432   0.877530 -1.499010 -1.061538  11.370221  294.448637
72  87900   0.263514   0.889676 -1.192676 -0.876923  11.383966  296.479342
64  87900   0.263514   0.889676 -1.192676 -0.876923  11.383966  296.479342
56  87600   0.243243   0.886640 -1.269259 -0.923077  11.380548  295.972972

Thống kê sau chuyển đổi:
           price  price_mms  price_mas  price_ss  price_rs  price_log  \
count    161.000    161.000    161.000   161.000   161.000    161.000   
mean   92572.050      0.579      0.937     0.000    -0.158     11.435   
std     3929.507      0.266      0.040     1.003     0.605      0.043   
min    84000.000      0.000      0.850    -2.188    -1.477     11.339   
25%    89300.000      0.358      0.904    -0.835    -0.662     11.400   
50%    93600.000      0.649      0.947     0.262     0.000     11.447   
75%    95800.

### Kết thúc Bước 3

Bạn đã minh họa đúng hai mục của slide: **Feature Engineering** và **Feature Transformation**. Sáu DataFrame gốc vẫn riêng biệt; hai DataFrame phụ (`df_coffee_gialai`, `df_coffee_scale`) chỉ dành cho bài thực hành này. Chưa gộp file, chưa One-Hot/Label/Ordinal Encoding (mục 4), và chưa chọn đặc trưng (mục 5).

**Lưu ý để dùng ở các chương sau:** `fit_transform()` ở đây chỉ để học cách chuẩn hóa. Khi thực sự đánh giá mô hình dự đoán, bộ biến đổi phải được fit trên dữ liệu huấn luyện, không fit cả dữ liệu dùng để đánh giá.


# Bước 4 – Mã hóa dữ liệu phân loại (Categorical Encoding)

**Vị trí:** Bước 4 được thêm **ngay sau mục “Kết thúc Bước 3”** của notebook. Chạy từ **Bước 4 – Cell 1** tới **Cell 5** theo thứ tự.

Theo slide DS111, chúng ta thực hành đúng 3 phương pháp: **Label Encoding, Ordinal Encoding, One-Hot Encoding**. Không gộp 6 CSV và không xây dựng mô hình dự đoán.

- **Label Encoding:** mỗi nhãn được gán một số nguyên; số này **không biểu thị thứ tự thực tế** nếu nhãn vốn không có thứ tự.
- **Ordinal Encoding:** chỉ áp dụng với phân loại **có thứ tự xác định trước**.
- **One-Hot Encoding:** phù hợp với tên địa phương, loại thời tiết, loại nhiên liệu (không có thứ tự tự nhiên).

Ta tiếp tục giữ nguyên 6 DataFrame riêng và tạo DataFrame phụ cho các ví dụ cần thiết.

### Bước 4 – Cell 1: Chuẩn bị dữ liệu cà phê nội địa để mã hóa

**Đã sửa lỗi KeyError**: Tạo trực tiếp `month`, `day_of_week`, `change_missing` **trong DataFrame phụ `df_coffee_encoding`**, sau khi lọc cà phê. Như vậy không cần giả định rằng ba cột đã tồn tại trong `df_nong_san`. Các phép biến đổi đều thuộc Chương 2: `.dt.month`, `.dt.dayofweek`, `isna().astype(int)`.

Chọn **cà phê nội địa**, giá `VND/kg`, nguồn `giacaphe.com`, ở 4 địa phương. Giữ nguyên `df_nong_san` và các DataFrame khác. Các Cell tiếp theo của Bước 4 và 5 dùng `df_coffee_encoding`.


In [93]:
# Bước 4 – Cell 1: Lọc cà phê bằng những cột đã có trong file gốc
# (không yêu cầu month, day_of_week, change_missing có trước)
df_coffee_encoding = df_nong_san.loc[
    (df_nong_san["commodity"] == "coffee")
    & (df_nong_san["type"] == "domestic")
    & (df_nong_san["source"] == "giacaphe.com")
    & (df_nong_san["unit"] == "VND/kg")
    & (df_nong_san["region"].isin(["Gia Lai", "Lâm Đồng", "Đắk Lắk", "Đắk Nông"]))
    & (df_nong_san["price"] > 0),
    ["date", "region", "price", "change"]
].copy()

# Tạo đặc trưng từ ngày tháng (Feature Engineering – Bước 3)
df_coffee_encoding["date"] = pd.to_datetime(df_coffee_encoding["date"])
df_coffee_encoding["month"] = df_coffee_encoding["date"].dt.month
df_coffee_encoding["day_of_week"] = df_coffee_encoding["date"].dt.dayofweek

# Tạo cờ báo thiếu (Missingness Indicator – Bước 1)
df_coffee_encoding["change_missing"] = (
    df_coffee_encoding["change"].isna().astype(int)
)

# Cột change chỉ dùng để tạo cờ, không đưa vào bảng mã hóa
df_coffee_encoding = df_coffee_encoding[
    ["date", "region", "month", "day_of_week", "change_missing", "price"]
]

df_coffee_encoding = df_coffee_encoding.sort_values(
    ["date", "region"]
).reset_index(drop=True)

print("Số bản ghi:", len(df_coffee_encoding))
print("Số ngày:", df_coffee_encoding["date"].nunique())
print("Số dòng trùng date + region:", df_coffee_encoding.duplicated(["date", "region"]).sum())
print(df_coffee_encoding["region"].value_counts())
print(df_coffee_encoding.head())

Số bản ghi: 644
Số ngày: 161
Số dòng trùng date + region: 0
region
Gia Lai     161
Lâm Đồng    161
Đắk Lắk     161
Đắk Nông    161
Name: count, dtype: int64[pyarrow]
        date    region  month  day_of_week  change_missing  price
0 2026-05-20   Gia Lai      5            2               1  86100
1 2026-05-20  Lâm Đồng      5            2               1  85600
2 2026-05-20   Đắk Lắk      5            2               1  86100
3 2026-05-20  Đắk Nông      5            2               1  86300
4 2026-05-22   Gia Lai      5            4               1  86700


C:\Users\lehuu\AppData\Local\Temp\ipykernel_9476\3616498221.py:16: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_coffee_encoding["day_of_week"] = df_coffee_encoding["date"].dt.dayofweek


### Bước 4 – Cell 2: Label Encoding cho trạng thái thời tiết

Dùng `LabelEncoder` trên `conditions` của `df_weather`. Vì trạng thái thời tiết **không có thứ tự từ thấp đến cao**, các số 0, 1, 2, ... chỉ là **mã nhãn để học thao tác**, không phải mức độ mưa/nắng.

In [94]:
from sklearn.preprocessing import LabelEncoder

label_e = LabelEncoder()
df_weather["conditions_le"] = label_e.fit_transform(df_weather["conditions"])

print("Bảng mã nhãn:")
for label, number in zip(label_e.classes_, label_e.transform(label_e.classes_)):
    print(label, "->", number)

print(df_weather[["conditions", "conditions_le"]].head())

Bảng mã nhãn:
bầu trời quang đãng -> 0
mây cụm -> 1
mây rải rác -> 2
mây thưa -> 3
mây đen u ám -> 4
mưa cường độ nặng -> 5
mưa nhẹ -> 6
mưa vừa -> 7
     conditions  conditions_le
0       mây cụm              1
1  mây đen u ám              4
2  mây đen u ám              4
3  mây đen u ám              4
4  mây đen u ám              4


### Bước 4 – Cell 3: Ordinal Encoding cho dữ liệu có thứ tự

Trong 6 CSV hiện tại **không có cột phân loại nào được xác nhận có thứ tự như Yếu → TB → Khá → Giỏi → SX**. Vì vậy, dùng đúng ví dụ `xep_loai` trong slide trên một DataFrame minh họa riêng. **Không dùng OrdinalEncoder cho `region`, `conditions`, `fuel_type` hoặc `zone`**.

In [95]:
from sklearn.preprocessing import OrdinalEncoder

df_ordinal_example = pd.DataFrame({
    "xep_loai": ["Yếu", "TB", "Khá", "Giỏi", "SX"]
})

ord_cats = [["Yếu", "TB", "Khá", "Giỏi", "SX"]]
ord_enc = OrdinalEncoder(categories=ord_cats)

df_ordinal_example["xep_loai_ord"] = ord_enc.fit_transform(
    df_ordinal_example[["xep_loai"]]
).ravel().astype(int)

print(df_ordinal_example)

  xep_loai  xep_loai_ord
0      Yếu             0
1       TB             1
2      Khá             2
3     Giỏi             3
4       SX             4


### Bước 4 – Cell 4: One-Hot Encoding cho địa phương

`region` không có thứ tự tự nhiên, nên dùng `OneHotEncoder` như slide. Sau khi mã hóa, mỗi vùng là một cột 0/1. `handle_unknown="ignore"` là tham số đã xuất hiện trong slide. Khi `pd.concat()`, hai bảng cần có chỉ số dòng phù hợp; Cell 1 đã dùng `reset_index(drop=True)` để đảm bảo điều đó.

In [96]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
region_encoded = ohe.fit_transform(df_coffee_encoding[["region"]])

region_columns = ohe.get_feature_names_out(["region"])
df_region_onehot = pd.DataFrame(
    region_encoded,
    columns=region_columns,
    index=df_coffee_encoding.index
).astype(int)

df_coffee_encoded = pd.concat(
    [df_coffee_encoding, df_region_onehot],
    axis=1
)

print(df_coffee_encoded[["region"] + list(region_columns)].head(8))

     region  region_Gia Lai  region_Lâm Đồng  region_Đắk Lắk  region_Đắk Nông
0   Gia Lai               1                0               0                0
1  Lâm Đồng               0                1               0                0
2   Đắk Lắk               0                0               1                0
3  Đắk Nông               0                0               0                1
4   Gia Lai               1                0               0                0
5  Lâm Đồng               0                1               0                0
6   Đắk Lắk               0                0               1                0
7  Đắk Nông               0                0               0                1


### Bước 4 – Cell 5: Kiểm tra kết quả mã hóa

Mỗi bản ghi cà phê chỉ thuộc **một địa phương**, nên tổng bốn cột One-Hot trên mỗi dòng phải bằng 1. Chúng ta vẫn giữ cột `region` để kiểm tra dễ dàng. Các cột mã hóa ở đây được tạo để thực hành; không nên dùng đồng thời `conditions_le` và One-Hot thời tiết trong một mô hình mà không xem xét ý nghĩa.

In [97]:
print("Số nhãn thời tiết:", df_weather["conditions_le"].nunique())
print("Số cột One-Hot địa phương:", len(region_columns))
print("Số dòng có tổng One-Hot khác 1:",
      (df_region_onehot.sum(axis=1) != 1).sum())
print("Số ô thiếu trong các cột One-Hot:", df_region_onehot.isna().sum().sum())
print("Kích thước dữ liệu cà phê đã mã hóa:", df_coffee_encoded.shape)

Số nhãn thời tiết: 8
Số cột One-Hot địa phương: 4
Số dòng có tổng One-Hot khác 1: 0
Số ô thiếu trong các cột One-Hot: 0
Kích thước dữ liệu cà phê đã mã hóa: (644, 10)


# Bước 5 – Lựa chọn đặc trưng (Feature Selection)

**Vị trí:** Bước 5 được thêm **ngay sau Bước 4 – Cell 5**. Chạy lần lượt **Bước 5 – Cell 1 → Cell 6**.

Theo slide, chọn đặc trưng để giảm nhiễu và tránh sử dụng các cột không phù hợp. Để **chưa học sang Chương 5**, ta tập trung vào các phương pháp thống kê có trong slide:

1. **Kiến thức nghiệp vụ (Domain):** chỉ giữ biến có ý nghĩa với mục tiêu phân tích giá cà phê; loại các cột `id`, `source_id`, `fetched_at` khỏi danh sách biến giải thích.
2. **Low Variance:** loại các cột có phương sai bằng 0.
3. **Correlation (Pearson):** xem mối liên hệ giữa biến số và giá cà phê.
4. **ANOVA:** minh họa so sánh giá trung bình giữa 4 địa phương.
5. **Mutual Information:** tham khảo mức thông tin của từng biến đối với giá.

**Chi-square** trong slide dùng cho dữ liệu phân loại. Vì mục tiêu của ví dụ này là **giá `price` dạng số liên tục**, ta **không áp Chi-square** chỉ để làm đủ kỹ thuật. Các phương pháp dựa trên mô hình (Decision Tree, Random Forest, XGBoost, RFE, Lasso) được nêu trong slide nhưng **không triển khai ở bước này**, tránh chuyển sang huấn luyện và đánh giá mô hình.

**Quan trọng:** Các con số dưới đây chỉ minh họa cách chọn biến trên dữ liệu hiện có. Không được xem chúng là kiểm chứng khả năng dự đoán hoặc bằng chứng nhân quả.

### Bước 5 – Cell 1: Xác định biến mục tiêu và đặc trưng ứng viên

- **Biến mục tiêu:** `price` (giá cà phê nội địa, VND/kg).
- **Đặc trưng ứng viên:** `month`, `day_of_week`, `change_missing` và 4 cột One-Hot của `region`.
- Không đưa `id`, `source_id`, `fetched_at` vào `X`: đó là thông tin định danh/thu thập. `date` được giữ riêng để mô tả thời gian, không trực tiếp coi như số đầu vào.
- Không sử dụng `price_ma7`, `price_mms`, ... để giải thích **chính giá `price` cùng thời điểm**, vì chúng được tính từ `price` và có thể gây rò rỉ thông tin về biến mục tiêu.

Ta dùng **cùng mẫu 644 dòng** từ Bước 4.

In [98]:
feature_columns = ["month", "day_of_week", "change_missing"] + list(region_columns)

X_coffee = df_coffee_encoded[feature_columns].copy()
y_coffee = df_coffee_encoded["price"].copy()

print("Các đặc trưng ứng viên:", feature_columns)
print("Kích thước X:", X_coffee.shape)
print("Kích thước y:", y_coffee.shape)
print("Giá trị thiếu trong X:", X_coffee.isna().sum().sum())

Các đặc trưng ứng viên: ['month', 'day_of_week', 'change_missing', 'region_Gia Lai', 'region_Lâm Đồng', 'region_Đắk Lắk', 'region_Đắk Nông']
Kích thước X: (644, 7)
Kích thước y: (644,)
Giá trị thiếu trong X: 0


### Bước 5 – Cell 2: Loại đặc trưng có phương sai thấp (Low Variance)

Phương sai bằng **0** nghĩa là toàn bộ dòng có cùng một giá trị, cột không giúp phân biệt các quan sát. Dùng `VarianceThreshold(threshold=0.0)` để bỏ **chỉ cột hằng**. Đặc trưng `change_missing` có thể là ví dụ thực tế vì giá trị `change` của mẫu cà phê nguồn này đều thiếu.

In [99]:
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.0)

X_coffee_selected = pd.DataFrame(
    vt.fit_transform(X_coffee),
    columns=X_coffee.columns[vt.get_support()],
    index=X_coffee.index
)

df_variance = pd.DataFrame({
    "feature": X_coffee.columns,
    "variance": vt.variances_,
    "keep": vt.get_support()
})

print(df_variance)
print("Các cột giữ lại:", X_coffee_selected.columns.tolist())

           feature  variance   keep
0            month  2.762471   True
1      day_of_week  4.136492   True
2   change_missing  0.000000  False
3   region_Gia Lai  0.187500   True
4  region_Lâm Đồng  0.187500   True
5   region_Đắk Lắk  0.187500   True
6  region_Đắk Nông  0.187500   True
Các cột giữ lại: ['month', 'day_of_week', 'region_Gia Lai', 'region_Lâm Đồng', 'region_Đắk Lắk', 'region_Đắk Nông']


### Bước 5 – Cell 3: Phân tích tương quan Pearson

Tính tương quan giữa các đặc trưng **đã qua Low Variance** và `price`. Giá trị từ -1 đến 1; dấu cho biết chiều liên hệ tuyến tính và trị tuyệt đối cho biết độ mạnh. Pearson **không chứng minh quan hệ nhân quả**; các cột 0/1 của vùng được tính như biến chỉ báo.

In [100]:
df_corr_input = pd.concat(
    [X_coffee_selected, y_coffee.rename("price")],
    axis=1
)

corr_price = df_corr_input.corr(numeric_only=True)["price"].drop("price")

df_correlation = pd.DataFrame({
    "pearson": corr_price,
    "abs_pearson": corr_price.abs()
}).sort_values("abs_pearson", ascending=False)

print(df_correlation.round(4))

                 pearson  abs_pearson
month             0.1802       0.1802
region_Lâm Đồng  -0.0626       0.0626
region_Đắk Nông   0.0363       0.0363
region_Đắk Lắk    0.0136       0.0136
region_Gia Lai    0.0127       0.0127
day_of_week      -0.0008       0.0008


### Bước 5 – Cell 4: ANOVA – So sánh giá trung bình theo địa phương

Đây là minh họa **One-way ANOVA** trong slide với `region` là nhóm và `price` là giá trị số. **F** đo tỷ lệ biến thiên giữa nhóm so với trong nhóm; **p-value** là căn cứ tham khảo khi kiểm định. Cùng một ngày, giá ở bốn vùng có thể biến động liên quan nhau và chuỗi thời gian không hoàn toàn độc lập, nên **không kết luận chắc chắn về khác biệt có ý nghĩa thống kê chỉ từ phép tính minh họa này**.

In [101]:
from scipy.stats import f_oneway

price_gialai = df_coffee_encoded.loc[
    df_coffee_encoded["region"] == "Gia Lai", "price"
]
price_lamdong = df_coffee_encoded.loc[
    df_coffee_encoded["region"] == "Lâm Đồng", "price"
]
price_daklak = df_coffee_encoded.loc[
    df_coffee_encoded["region"] == "Đắk Lắk", "price"
]
price_daknong = df_coffee_encoded.loc[
    df_coffee_encoded["region"] == "Đắk Nông", "price"
]

f_stat, p_value = f_oneway(
    price_gialai, price_lamdong, price_daklak, price_daknong
)

print("Giá trung bình từng địa phương:")
print(df_coffee_encoded.groupby("region")["price"].mean().round(2))
print("F-statistic:", round(f_stat, 4))
print("p-value:", round(p_value, 6))

Giá trung bình từng địa phương:
region
Gia Lai     92572.05
Lâm Đồng    92060.86
Đắk Lắk     92578.26
Đắk Nông    92732.92
Name: price, dtype: float64
F-statistic: 0.8971
p-value: 0.442333


### Bước 5 – Cell 5: Mutual Information (đã có trong slide)

`mutual_info_regression()` đo lượng thông tin mà đặc trưng cung cấp về **biến mục tiêu số**. Ở đây `month`, `day_of_week` và các cột vùng One-Hot đều chỉ nhận số lượng giá trị rời rạc, nên đặt `discrete_features=True`. Chỉ dùng điểm để **tham khảo**, không tự kết luận biến điểm thấp là vô ích hay biến điểm cao gây ra biến động giá.

In [102]:
from sklearn.feature_selection import mutual_info_regression

mi_values = mutual_info_regression(
    X_coffee_selected,
    y_coffee,
    discrete_features=True,
    random_state=0
)

df_mutual_info = pd.DataFrame({
    "feature": X_coffee_selected.columns,
    "mutual_information": mi_values
}).sort_values("mutual_information", ascending=False)

print(df_mutual_info.round(4))

           feature  mutual_information
0            month              1.1216
1      day_of_week              0.1524
5  region_Đắk Nông              0.1005
3  region_Lâm Đồng              0.0873
2   region_Gia Lai              0.0151
4   region_Đắk Lắk              0.0000


### Bước 5 – Cell 6: Tổng kết tập đặc trưng đã chọn

Lựa chọn **bước đầu** theo kiến thức nghiệp vụ + Low Variance: bỏ các cột định danh, cột mục tiêu ở phía X, và cột hằng (`change_missing`). Pearson, ANOVA, Mutual Information được dùng **để tìm hiểu đặc trưng**, không áp dụng một ngưỡng cắt tùy ý. Vì **chưa sang Chương 5**, ta **chưa huấn luyện mô hình** hay chọn đặc trưng tối ưu cho dự đoán.

In [103]:
selected_features = X_coffee_selected.columns.tolist()

df_coffee_prepared = pd.concat(
    [df_coffee_encoded[["date", "region", "price"]], X_coffee_selected],
    axis=1
)

print("Biến mục tiêu: price")
print("Các biến ứng viên đã qua Low Variance:", selected_features)
print("Kích thước tập thực hành:", df_coffee_prepared.shape)
print("Cột change_missing đã bị loại khỏi X:", "change_missing" not in selected_features)
print("Số giá trị thiếu trong X:", X_coffee_selected.isna().sum().sum())
print(df_coffee_prepared.head())

Biến mục tiêu: price
Các biến ứng viên đã qua Low Variance: ['month', 'day_of_week', 'region_Gia Lai', 'region_Lâm Đồng', 'region_Đắk Lắk', 'region_Đắk Nông']
Kích thước tập thực hành: (644, 9)
Cột change_missing đã bị loại khỏi X: True
Số giá trị thiếu trong X: 0
        date    region  price  month  day_of_week  region_Gia Lai  \
0 2026-05-20   Gia Lai  86100      5            2               1   
1 2026-05-20  Lâm Đồng  85600      5            2               0   
2 2026-05-20   Đắk Lắk  86100      5            2               0   
3 2026-05-20  Đắk Nông  86300      5            2               0   
4 2026-05-22   Gia Lai  86700      5            4               1   

   region_Lâm Đồng  region_Đắk Lắk  region_Đắk Nông  
0                0               0                0  
1                1               0                0  
2                0               1                0  
3                0               0                1  
4                0               0                

## Kết thúc Bước 4 và Bước 5 – Kiểm tra phạm vi học tập

**Bước 4 đã thực hành:** `LabelEncoder` trên `conditions`, `OrdinalEncoder` với **ví dụ thứ bậc trong slide**, `OneHotEncoder` trên `region` cà phê. Không dùng nhãn LabelEncoder để suy ra thứ tự thời tiết.

**Bước 5 đã thực hành:** lựa chọn theo nghiệp vụ, **Low Variance**, **Pearson**, **One-way ANOVA**, **Mutual Information** — đều được nêu trong Chương 2.

**Chưa thực hiện:** Chi-square (mục tiêu hiện là giá số liên tục), các thuật toán chọn đặc trưng dựa trên mô hình, gộp 6 CSV, EDA theo Chương 4, Dashboard và dự đoán theo Chương 5.

**Các DataFrame gốc vẫn tách riêng:** `df_nong_san`, `df_weather`, `df_exchange`, `df_fuel`, `df_commodities_api`, `df_commodities_history`. Các bảng `df_coffee_encoding`, `df_coffee_encoded`, `df_coffee_prepared` dùng để học Chương 2; **không phải file tích hợp cuối cùng**.

**Lưu ý cho chương học sau:** Mọi bước `fit()` / `fit_transform()` trong notebook này mới là **thực hành trên dữ liệu đã có**; khi làm mô hình cần học bộ biến đổi/chọn đặc trưng từ tập huấn luyện để tránh rò rỉ thông tin.

# Hoàn thiện Chương 2 – Tích hợp dữ liệu và xuất CSV

**Vị trí:** Phần này nằm **ngay sau mục “Kết thúc Bước 4 và Bước 5”**, không thay thế các Cell trước.

Đây là **bài thực hành tổng hợp** những thao tác đã học ở Chương 2: xử lý thiếu (median, giá trị cố định), định dạng, tạo đặc trưng, giữ mã hóa phù hợp và xem xét đặc trưng trước khi lưu dữ liệu. **Không** huấn luyện mô hình, không thực hiện EDA chuyên sâu hay Dashboard.

**Bài toán:** Phân tích giá cà phê nội địa tại Gia Lai, Lâm Đồng, Đắk Lắk và Đắk Nông theo từng ngày. Kết hợp thời tiết, tỷ giá USD/VND, dầu Diesel, giá Robusta hợp đồng cố định 11/2027 và dầu Brent.

**Mục tiêu xuất:** `../data/processed/coffee_analytics_clean.csv` (**15 cột**: `date`, biến mục tiêu và **13 biến/đặc trưng còn lại**). Đường dẫn có cùng quy tắc với `../data/raw/` của notebook. Chạy notebook **từ đầu theo thứ tự** trước khi đến phần này.

> Việc ghi file vào `data/processed/` **chưa tự động tải lên GitHub.com**; xem Cell cuối để biết cách commit/push từ bản sao Git trên máy.

### Tổng hợp – Cell 1: Chọn dữ liệu nông sản làm bảng trung tâm

Lấy cà phê **nội địa**, đơn vị **VND/kg**, nguồn `giacaphe.com` và bốn địa phương. Không gộp với hồ tiêu hoặc loại tiền tệ khác. Giữ một dòng cho mỗi cặp `date + region`.

In [104]:
# Cà phê nội địa (bảng trung tâm)
df_coffee_base = df_nong_san.loc[
    (df_nong_san["commodity"] == "coffee")
    & (df_nong_san["type"] == "domestic")
    & (df_nong_san["unit"] == "VND/kg")
    & (df_nong_san["source"] == "giacaphe.com")
    & (df_nong_san["region"].isin(["Gia Lai", "Lâm Đồng", "Đắk Lắk", "Đắk Nông"]))
    & (df_nong_san["price"] > 0),
    ["date", "region", "price"]
].copy()

df_coffee_base = df_coffee_base.sort_values(["date", "region"])
df_coffee_base = df_coffee_base.drop_duplicates(subset=["date", "region"])
df_coffee_base = df_coffee_base.rename(columns={"price": "coffee_price_vnd_kg"})

print("Bảng trung tâm:", df_coffee_base.shape)
print("Ngày:", df_coffee_base["date"].nunique())
print("Trùng date + region:", df_coffee_base.duplicated(["date", "region"]).sum())
print(df_coffee_base.head())

Bảng trung tâm: (644, 3)
Ngày: 161
Trùng date + region: 0
         date    region  coffee_price_vnd_kg
80 2026-05-20   Gia Lai                86100
81 2026-05-20  Lâm Đồng                85600
82 2026-05-20   Đắk Lắk                86100
83 2026-05-20  Đắk Nông                86300
76 2026-05-22   Gia Lai                86700


### Tổng hợp – Cell 2: Chuẩn bị dữ liệu thời tiết

`target_region` trong file thời tiết tương ứng với `region` trong file nông sản. Không gộp theo `city` vì `Pleiku` và `Gia Lai` là hai cấp địa lý khác nhau.

In [105]:
# Thời tiết: ngày + địa phương
df_weather_join = df_weather[
    ["date", "target_region", "temperature", "humidity", "conditions"]
].copy()

df_weather_join = df_weather_join.rename(columns={
    "target_region": "region",
    "temperature": "temperature_c",
    "humidity": "humidity_pct",
    "conditions": "weather_conditions"
})

df_weather_join = df_weather_join.drop_duplicates(subset=["date", "region"])

print("Thời tiết:", df_weather_join.shape)
print("Trùng date + region:", df_weather_join.duplicated(["date", "region"]).sum())

Thời tiết: (854, 5)
Trùng date + region: 0


### Tổng hợp – Cell 3: Chuẩn bị dữ liệu tỷ giá USD/VND

Chọn **một nguồn thống nhất** là `vietcombank.com.vn`, không lấy cả nguồn `cafef.vn` cho cùng ngày. Giữ tỷ giá chuyển khoản (`transfer`) và chênh lệch bán–mua (`spread`) đã tạo ở Bước 3.

In [106]:
# Tỷ giá: một nguồn, một dòng/ngày
df_exchange_join = df_exchange.loc[
    (df_exchange["currency"] == "USD")
    & (df_exchange["source"] == "vietcombank.com.vn"),
    ["date", "transfer", "spread"]
].copy()

df_exchange_join = df_exchange_join.drop_duplicates(subset=["date"])
df_exchange_join = df_exchange_join.rename(columns={
    "transfer": "usd_transfer_vnd",
    "spread": "usd_spread_vnd"
})

print("Tỷ giá:", df_exchange_join.shape)
print("Trùng ngày:", df_exchange_join.duplicated("date").sum())

Tỷ giá: (180, 3)
Trùng ngày: 0


### Tổng hợp – Cell 4: Chuẩn bị dữ liệu giá nhiên liệu

Chọn loại **`Dầu Diesel`**, nguồn `vnexpress.net`. File nhiên liệu không có cột đơn vị đo nên giữ tên cột là `diesel_price` (không tự khẳng định đơn vị).

In [107]:
# Nhiên liệu: một loại và một nguồn cụ thể
df_fuel_join = df_fuel.loc[
    (df_fuel["fuel_type"] == "Dầu Diesel")
    & (df_fuel["source"] == "vnexpress.net"),
    ["date", "price"]
].copy()

df_fuel_join = df_fuel_join.drop_duplicates(subset=["date"])
df_fuel_join = df_fuel_join.rename(columns={"price": "diesel_price"})

print("Nhiên liệu:", df_fuel_join.shape)
print("Trùng ngày:", df_fuel_join.duplicated("date").sum())

Nhiên liệu: (167, 2)
Trùng ngày: 0


### Tổng hợp – Cell 5: Chuẩn bị giá hàng hóa thế giới từ 2 file CSV

`commodities_history_v2.csv` là **tập con trùng** của `commodities_api_raw_v2.csv` đối với các chuỗi được chọn. Dùng `pd.concat()` rồi `drop_duplicates()` để **không nhân đôi bản ghi**.

- **Robusta:** chọn **một hợp đồng cố định `Coffee Robusta (ICE London) 11/2027`**, đơn vị `USD/tonne`; **không** lấy trung bình các kỳ hạn khác nhau.
- **Brent:** chọn `Crude Oil Brent (ICE front-month)`, đơn vị `USD/barrel`; đây là chuỗi front-month như nguồn cung cấp, không phải cùng một kỳ hạn cố định.

In [108]:
# Ghép hai nguồn trước khi loại bản ghi trùng
df_commodities_all = pd.concat(
    [df_commodities_api, df_commodities_history], ignore_index=True
)

# Robusta: cố định hợp đồng 11/2027
df_robusta_join = df_commodities_all.loc[
    (df_commodities_all["name"] == "Coffee Robusta (ICE London) 11/2027")
    & (df_commodities_all["unit"] == "USD/tonne"),
    ["date", "price"]
].copy()
robusta_before = len(df_robusta_join)
df_robusta_join = df_robusta_join.drop_duplicates(subset=["date"], keep="first")
df_robusta_join = df_robusta_join.rename(columns={
    "price": "robusta_price_usd_tonne"
})

# Brent: chuỗi front-month như trong nguồn
df_brent_join = df_commodities_all.loc[
    (df_commodities_all["name"] == "Crude Oil Brent (ICE front-month)")
    & (df_commodities_all["unit"] == "USD/barrel"),
    ["date", "price"]
].copy()
brent_before = len(df_brent_join)
df_brent_join = df_brent_join.drop_duplicates(subset=["date"], keep="first")
df_brent_join = df_brent_join.rename(columns={
    "price": "brent_price_usd_barrel"
})

print("Robusta trước/sau bỏ trùng:", robusta_before, len(df_robusta_join))
print("Brent trước/sau bỏ trùng:", brent_before, len(df_brent_join))

Robusta trước/sau bỏ trùng: 330 165
Brent trước/sau bỏ trùng: 298 149


### Tổng hợp – Cell 6: Ghép dữ liệu từ các nguồn bằng `pd.merge()`

Ghép **left join** để không mất dòng nông sản khi một nguồn phụ không có giá trị cùng ngày. Chọn mỗi nguồn phụ một dòng/ngày (hoặc ngày + địa phương) trước khi ghép để tránh **nhân bản dòng**.

In [109]:
# Ghép lần lượt, giữ toàn bộ dòng của bảng cà phê
df_coffee_merged = pd.merge(
    df_coffee_base, df_weather_join,
    on=["date", "region"], how="left"
)

df_coffee_merged = pd.merge(
    df_coffee_merged, df_exchange_join,
    on="date", how="left"
)

df_coffee_merged = pd.merge(
    df_coffee_merged, df_fuel_join,
    on="date", how="left"
)

df_coffee_merged = pd.merge(
    df_coffee_merged, df_robusta_join,
    on="date", how="left"
)

df_coffee_merged = pd.merge(
    df_coffee_merged, df_brent_join,
    on="date", how="left"
)

print("Số dòng cà phê gốc:", len(df_coffee_base))
print("Số dòng sau ghép:", len(df_coffee_merged))
print("Trùng date + region:", df_coffee_merged.duplicated(["date", "region"]).sum())
print("Số giá trị thiếu sau ghép:")
print(df_coffee_merged.isna().sum())

assert len(df_coffee_merged) == len(df_coffee_base), "Ghép dữ liệu làm thay đổi số dòng."
assert df_coffee_merged.duplicated(["date", "region"]).sum() == 0

Số dòng cà phê gốc: 644
Số dòng sau ghép: 644
Trùng date + region: 0
Số giá trị thiếu sau ghép:
date                         0
region                       0
coffee_price_vnd_kg          0
temperature_c                3
humidity_pct                 3
weather_conditions           3
usd_transfer_vnd             8
usd_spread_vnd               8
diesel_price                 0
robusta_price_usd_tonne     52
brent_price_usd_barrel     112
dtype: int64


### Tổng hợp – Cell 7: Feature Engineering và Missingness Indicator

Áp dụng kỹ thuật trong slide:

- Tách `date` thành `month` và `day_of_week`.
- Tạo **giá trung bình 7 quan sát trước đó** theo từng địa phương từ `price`, dùng `rolling()`; dùng `shift(1)` đơn giản để **không sử dụng giá của chính ngày cần phân tích**.
- Đánh dấu `market_missing = 1` nếu trước khi điền khuyết có thiếu ít nhất một biến tỷ giá/Robusta/Brent.

7 quan sát trước **không nhất thiết** là 7 ngày lịch liên tục. Mỗi địa phương có 7 dòng đầu chưa đủ lịch sử, nên sẽ loại các dòng đó bằng `dropna()` theo phương pháp Deletion ở phần xử lý thiếu.

In [110]:
# Sắp xếp trước khi tính rolling theo từng địa phương
df_coffee_merged = df_coffee_merged.sort_values(
    ["region", "date"]
).reset_index(drop=True)

# Đặc trưng ngày tháng
df_coffee_merged["month"] = df_coffee_merged["date"].dt.month
df_coffee_merged["day_of_week"] = df_coffee_merged["date"].dt.dayofweek

# Thiếu thông tin thị trường trước khi điền giá trị
df_coffee_merged["market_missing"] = df_coffee_merged[
    ["usd_transfer_vnd", "usd_spread_vnd", "robusta_price_usd_tonne", "brent_price_usd_barrel"]
].isna().any(axis=1).astype(int)

# Trung bình đúng 7 quan sát trước đó của cùng địa phương
df_coffee_merged["coffee_price_ma7_prev"] = (
    df_coffee_merged.groupby("region")["coffee_price_vnd_kg"]
    .transform(lambda s: s.shift(1).rolling(window=7, min_periods=7).mean())
)

print("Dòng chưa đủ 7 quan sát trước:", df_coffee_merged["coffee_price_ma7_prev"].isna().sum())
print("Dòng thiếu ít nhất một giá thị trường:", df_coffee_merged["market_missing"].sum())

Dòng chưa đủ 7 quan sát trước: 28
Dòng thiếu ít nhất một giá thị trường: 160


C:\Users\lehuu\AppData\Local\Temp\ipykernel_9476\2324820389.py:8: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_coffee_merged["day_of_week"] = df_coffee_merged["date"].dt.dayofweek


### Tổng hợp – Cell 8: Xử lý dữ liệu khuyết sau ghép

Chỉ dùng **các cách đã học**: `dropna()` (xóa những dòng chưa đủ 7 giá trước), `median` (thay giá trị thiếu ở cột số), `fillna('Không rõ')` (cột phân loại), và cờ `market_missing` đã tạo.

- Thời tiết: điền median **theo địa phương**; ít dòng thiếu nên không làm xáo trộn toàn bộ dữ liệu.
- Tỷ giá/Robusta/Brent/Diesel: điền median theo **chính cột** để còn dùng dataset trong bài thực hành; giá điền chỉ là **ước lượng thống kê**, không phải giá thị trường được ghi nhận.
- Khi đến Chương 5, việc điền giá trị khuyết và Scaling cần xem xét lại theo dữ liệu huấn luyện, tránh sử dụng thông tin tương lai để đánh giá dự đoán. Notebook hiện tại **chưa huấn luyện mô hình**.

In [111]:
# Bỏ 7 dòng đầu mỗi địa phương do không đủ 7 quan sát trước
df_coffee_merged = df_coffee_merged.dropna(
    subset=["coffee_price_ma7_prev"]
).copy()

# Điền khuyết thời tiết bằng median của cùng địa phương
df_coffee_merged["temperature_c"] = df_coffee_merged["temperature_c"].fillna(
    df_coffee_merged.groupby("region")["temperature_c"].transform("median")
)
df_coffee_merged["humidity_pct"] = df_coffee_merged["humidity_pct"].fillna(
    df_coffee_merged.groupby("region")["humidity_pct"].transform("median")
)
df_coffee_merged["weather_conditions"] = (
    df_coffee_merged["weather_conditions"].fillna("Không rõ")
)

# Điền khuyết số theo median của từng cột
df_coffee_merged["usd_transfer_vnd"] = df_coffee_merged["usd_transfer_vnd"].fillna(
    df_coffee_merged["usd_transfer_vnd"].median()
)
df_coffee_merged["usd_spread_vnd"] = df_coffee_merged["usd_spread_vnd"].fillna(
    df_coffee_merged["usd_spread_vnd"].median()
)
df_coffee_merged["diesel_price"] = df_coffee_merged["diesel_price"].fillna(
    df_coffee_merged["diesel_price"].median()
)
df_coffee_merged["robusta_price_usd_tonne"] = df_coffee_merged["robusta_price_usd_tonne"].fillna(
    df_coffee_merged["robusta_price_usd_tonne"].median()
)
df_coffee_merged["brent_price_usd_barrel"] = df_coffee_merged["brent_price_usd_barrel"].fillna(
    df_coffee_merged["brent_price_usd_barrel"].median()
)

print("Số dòng sau loại 7 quan sát đầu mỗi vùng:", len(df_coffee_merged))
print("Số giá trị thiếu còn lại:")
print(df_coffee_merged.isna().sum())

Số dòng sau loại 7 quan sát đầu mỗi vùng: 616
Số giá trị thiếu còn lại:
date                       0
region                     0
coffee_price_vnd_kg        0
temperature_c              0
humidity_pct               0
weather_conditions         0
usd_transfer_vnd           0
usd_spread_vnd             0
diesel_price               0
robusta_price_usd_tonne    0
brent_price_usd_barrel     0
month                      0
day_of_week                0
market_missing             0
coffee_price_ma7_prev      0
dtype: int64


### Tổng hợp – Cell 9: Lựa chọn 15 cột để xuất

Dữ liệu cuối có **15 cột**, gồm 1 khóa thời gian `date`, 1 biến mục tiêu `coffee_price_vnd_kg` và **13 đặc trưng ứng viên** (`region` cùng các biến thời tiết, thị trường và đặc trưng tạo thêm). Các cột `id`, `source_id`, `fetched_at` không có ý nghĩa dự báo trực tiếp và không được đưa vào CSV này.

Vẫn giữ `region` và `weather_conditions` dưới dạng nhãn chữ để thuận tiện Dashboard; Bước 4 đã có phần thực hành One-Hot/Label Encoding riêng. Các cột MinMax/Standard/Robust/Log/Sqrt cũng chỉ là minh họa, không thay thế giá gốc trong CSV.

In [112]:
# Bảng dữ liệu tích hợp cuối cùng: 15 cột
final_columns = [
    "date", "region", "coffee_price_vnd_kg",
    "temperature_c", "humidity_pct", "weather_conditions",
    "usd_transfer_vnd", "usd_spread_vnd", "diesel_price",
    "robusta_price_usd_tonne", "brent_price_usd_barrel",
    "month", "day_of_week", "coffee_price_ma7_prev", "market_missing"
]

df_dataset = df_coffee_merged[final_columns].copy()
df_dataset = df_dataset.sort_values(["date", "region"]).reset_index(drop=True)

print("Kích thước dataset:", df_dataset.shape)
print("Danh sách cột:", df_dataset.columns.tolist())
print(df_dataset.head())

Kích thước dataset: (616, 15)
Danh sách cột: ['date', 'region', 'coffee_price_vnd_kg', 'temperature_c', 'humidity_pct', 'weather_conditions', 'usd_transfer_vnd', 'usd_spread_vnd', 'diesel_price', 'robusta_price_usd_tonne', 'brent_price_usd_barrel', 'month', 'day_of_week', 'coffee_price_ma7_prev', 'market_missing']
        date    region  coffee_price_vnd_kg  temperature_c  humidity_pct  \
0 2026-05-28   Gia Lai                87700          24.61          78.0   
1 2026-05-28  Lâm Đồng                87200          20.57          83.0   
2 2026-05-28   Đắk Lắk                87700          25.45          75.0   
3 2026-05-28  Đắk Nông                87800          22.92          96.0   
4 2026-05-29   Gia Lai                89100          23.77          80.0   

  weather_conditions  usd_transfer_vnd  usd_spread_vnd  diesel_price  \
0           mây thưa           26143.0           280.0         27650   
1       mây đen u ám           26143.0           280.0         27650   
2       mây

### Tổng hợp – Cell 10: Kiểm tra Feature Selection và chất lượng dataset

Dùng **Low Variance** và **Pearson Correlation** như Bước 5, lần này kiểm tra ngay trên các biến số trong CSV cuối. `region` và `weather_conditions` giữ nguyên dạng chữ (có thể mã hóa ở giai đoạn dùng mô hình). Không tự động xóa những cột có tương quan thấp: đó chỉ là thông tin đánh giá ban đầu.

In [113]:
# Chỉ xem xét các đặc trưng số: không đưa biến mục tiêu vào X
numeric_features = [
    "temperature_c", "humidity_pct", "usd_transfer_vnd", "usd_spread_vnd",
    "diesel_price", "robusta_price_usd_tonne", "brent_price_usd_barrel",
    "month", "day_of_week", "coffee_price_ma7_prev", "market_missing"
]

from sklearn.feature_selection import VarianceThreshold

vt_final = VarianceThreshold(threshold=0.0)
vt_final.fit(df_dataset[numeric_features])

print("Đặc trưng số có phương sai khác 0:")
print(df_dataset[numeric_features].columns[vt_final.get_support()].tolist())

print()
print("Tương quan Pearson với giá cà phê (chỉ để tham khảo):")
print(
    df_dataset[numeric_features + ["coffee_price_vnd_kg"]]
    .corr(numeric_only=True)["coffee_price_vnd_kg"]
    .drop("coffee_price_vnd_kg")
    .round(3)
)

# Kiểm tra trước khi xuất
assert df_dataset.shape[1] == 15, "Dataset không có đúng 15 cột."
assert df_dataset.duplicated(["date", "region"]).sum() == 0, "Có dòng trùng date + region."
assert df_dataset["coffee_price_vnd_kg"].gt(0).all(), "Có giá cà phê không hợp lệ."
assert df_dataset.isna().sum().sum() == 0, "Dataset vẫn còn giá trị thiếu."
assert df_dataset["market_missing"].isin([0, 1]).all()
print()
print("Đã kiểm tra: 15 cột, không trùng khóa, giá dương, không còn NaN.")

Đặc trưng số có phương sai khác 0:
['temperature_c', 'humidity_pct', 'usd_transfer_vnd', 'usd_spread_vnd', 'diesel_price', 'robusta_price_usd_tonne', 'brent_price_usd_barrel', 'month', 'day_of_week', 'coffee_price_ma7_prev', 'market_missing']

Tương quan Pearson với giá cà phê (chỉ để tham khảo):
temperature_c             -0.066
humidity_pct               0.360
usd_transfer_vnd           0.140
usd_spread_vnd             0.627
diesel_price              -0.072
robusta_price_usd_tonne    0.769
brent_price_usd_barrel    -0.103
month                      0.088
day_of_week               -0.006
coffee_price_ma7_prev      0.924
market_missing             0.121
Name: coffee_price_vnd_kg, dtype: float64

Đã kiểm tra: 15 cột, không trùng khóa, giá dương, không còn NaN.


### Tổng hợp – Cell 11: Xuất CSV vào `data/processed/`

**Chèn ngay sau Cell 10.** Dùng `os.makedirs()` (Python cơ bản) để tạo thư mục nếu chưa có. Không dùng `Path`, không dùng dictionary hay vòng lặp để đọc file. Lưu bằng `utf-8-sig` để tiếng Việt hiển thị ổn khi mở file CSV trong Excel.

Nếu notebook nằm trong `notebooks/` và đã đọc dữ liệu từ `../data/raw/`, file xuất sẽ nằm ở **`data/processed/coffee_analytics_clean.csv`** trong cùng repository local.

In [114]:
import os

# Tạo thư mục data/processed (nếu chưa tồn tại)
os.makedirs("../data/processed", exist_ok=True)

# Chỉ định dạng ngày khi ghi CSV; DataFrame trong notebook vẫn giữ datetime
df_dataset_export = df_dataset.copy()
df_dataset_export["date"] = df_dataset_export["date"].dt.strftime("%Y-%m-%d")

output_csv = "../data/processed/coffee_analytics_clean.csv"
df_dataset_export.to_csv(output_csv, index=False, encoding="utf-8-sig")

# Đọc lại để xác nhận đã lưu đúng file
check_csv = pd.read_csv(output_csv)
print("Đã tạo:", os.path.abspath(output_csv))
print("Kích thước CSV:", check_csv.shape)
print("Số ô thiếu:", int(check_csv.isna().sum().sum()))
print("Trùng date + region:", int(check_csv.duplicated(["date", "region"]).sum()))
print(check_csv.head())

assert check_csv.shape == df_dataset_export.shape
assert check_csv.isna().sum().sum() == 0

Đã tạo: d:\Cpp\ds111\data\processed\coffee_analytics_clean.csv
Kích thước CSV: (616, 15)
Số ô thiếu: 0
Trùng date + region: 0
         date    region  coffee_price_vnd_kg  temperature_c  humidity_pct  \
0  2026-05-28   Gia Lai                87700          24.61          78.0   
1  2026-05-28  Lâm Đồng                87200          20.57          83.0   
2  2026-05-28   Đắk Lắk                87700          25.45          75.0   
3  2026-05-28  Đắk Nông                87800          22.92          96.0   
4  2026-05-29   Gia Lai                89100          23.77          80.0   

  weather_conditions  usd_transfer_vnd  usd_spread_vnd  diesel_price  \
0           mây thưa           26143.0           280.0         27650   
1       mây đen u ám           26143.0           280.0         27650   
2       mây đen u ám           26143.0           280.0         27650   
3            mây cụm           26143.0           280.0         27650   
4       mây đen u ám           26143.0           28

## Kết quả và hướng dẫn lưu trên GitHub

**Kết quả dự kiến với dữ liệu gốc hiện tại:** 644 dòng cà phê từ `giacaphe.com` (4 vùng × 161 ngày); sau khi loại 7 dòng đầu mỗi vùng để có đủ giá trung bình **7 quan sát trước**, còn **616 dòng × 15 cột**. `date + region` là khóa duy nhất. Ở dữ liệu cuối, các ô thiếu đã được xử lý theo cách đã học ở Chương 2.

**Từ khoá các cột (codebook):**

| Cột | Ý nghĩa |
|---|---|
| `date` | Ngày ghi nhận giá cà phê |
| `region` | Địa phương: Gia Lai, Lâm Đồng, Đắk Lắk, Đắk Nông |
| `coffee_price_vnd_kg` | **Biến mục tiêu**: giá cà phê nội địa, VND/kg |
| `temperature_c` | Nhiệt độ của thành phố đại diện, °C |
| `humidity_pct` | Độ ẩm của thành phố đại diện, % |
| `weather_conditions` | Nhãn thời tiết, giữ chuỗi để trực quan hóa |
| `usd_transfer_vnd` | Tỷ giá USD chuyển khoản (Vietcombank), VND/USD |
| `usd_spread_vnd` | Chênh lệch `sell - buy`, VND/USD |
| `diesel_price` | Giá `Dầu Diesel` theo VnExpress; **file gốc không ghi rõ đơn vị** |
| `robusta_price_usd_tonne` | Giá Robusta kỳ hạn **11/2027**, USD/tonne |
| `brent_price_usd_barrel` | Giá dầu Brent front-month, USD/barrel |
| `month` | Tháng trong năm (1–12) |
| `day_of_week` | Thứ trong tuần (0 = thứ Hai, 6 = Chủ nhật) |
| `coffee_price_ma7_prev` | Trung bình **7 giá cà phê quan sát trước ngày hiện tại** ở cùng địa phương |
| `market_missing` | 1 = ít nhất một trong các biến tỷ giá/Robusta/Brent ban đầu bị thiếu và đã được điền bằng median; 0 = đủ dữ liệu |

**Giới hạn dữ liệu:** Những ngày thiếu báo giá thị trường có thể là ngày không giao dịch. Việc điền median chỉ là cách **thực hành xử lý thiếu theo slide**, không thay thế dữ liệu quan sát thật. Khi xây mô hình ở Chương 5, phải đánh giá lại cách điền thiếu trên phần dữ liệu huấn luyện để tránh rò rỉ thông tin. Nhiệt độ/độ ẩm là từ thành phố đại diện, không phải đo cho cả tỉnh. Không suy luận quan hệ nhân quả từ việc ghép dữ liệu.

**Đẩy CSV lên GitHub:** Notebook chỉ ghi file vào **thư mục repository trên máy**. Mở Terminal tại **thư mục gốc repository** rồi chạy:

```bash
git add data/processed/coffee_analytics_clean.csv
git commit -m "Add processed DS111 coffee dataset"
git push
```

Nếu muốn lưu luôn notebook đã sửa vào GitHub, đặt notebook mới trong `notebooks/`, sau đó `git add` thêm đường dẫn notebook trước khi commit. Không cần tạo GitHub Action hoặc công cụ nâng cao.

**Dừng tại đây:** 5 nội dung của Chương 2 đã được thực hành và kết thúc bằng một CSV tích hợp. Các bước EDA, Dashboard, Modeling/Evaluation thuộc các chương sau.